# Practice 18 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: DateTime + MultiIndex
A library checkout DataFrame has been created for you.

1. Convert `checkout_date` to datetime. Extract `year` and `month`.
2. Add `days_ago` = days between `checkout_date` and `2026-03-16`. Find mean and median with NumPy.
3. Set a `(branch, genre)` MultiIndex. Sort it.
4. Retrieve all rows for `'Central'`.
5. Use `pd.IndexSlice` to get all `'Mystery'` rows across every branch.

In [8]:
# Starter data — don't change this
checkouts = pd.DataFrame({
    'branch':        ['Central', 'Central', 'Westside', 'Westside', 'Northgate', 'Northgate', 'Central', 'Westside'],
    'genre':         ['Mystery', 'Sci-Fi', 'Mystery', 'Romance', 'Sci-Fi', 'Mystery', 'Romance', 'Sci-Fi'],
    'checkout_date': ['2024-01-12', '2024-04-03', '2024-06-18', '2024-08-27',
                      '2024-10-05', '2025-02-14', '2025-05-20', '2025-09-09'],
    'copies_out': [14, 8, 22, 11, 5, 18, 9, 31],
    'holds':      [3, 0, 7, 2, 1, 5, 4, 9],
})

# Your code here
checkouts['checkout_date'] = pd.to_datetime(checkouts['checkout_date'])
checkouts['year'] = checkouts['checkout_date'].dt.year
checkouts['month'] = checkouts['checkout_date'].dt.month
checkouts['days_ago'] = (pd.to_datetime('2026-03-16')-checkouts['checkout_date']).dt.days
np.mean(checkouts['days_ago'])
np.median(checkouts['days_ago'])
checkouts = checkouts.set_index(['branch','genre']).sort_index()
checkouts
checkouts.loc['Central']
idx = pd.IndexSlice
checkouts.loc[idx[:,'Mystery'],:]


,,checkout_date,copies_out,holds,year,month,days_ago
branch,genre,,,,,,
Central,Mystery,2024-01-12,14,3,2024,1,794
Northgate,Mystery,2025-02-14,18,5,2025,2,395
Westside,Mystery,2024-06-18,22,7,2024,6,636


---
## Level 2 — Transformations

### Task 2: stack() / unstack()
A wide DataFrame of monthly energy usage (kWh) by building has been created for you.

1. Stack to long format. Store as `energy_long` (use `.to_frame('kwh')`)
2. Add `log_kwh` using NumPy
3. Add `high_usage`: `True` if `kwh` > 300 (use `np.where`)
4. Unstack back to wide format
5. Find the building with the highest mean usage using NumPy

In [12]:
# Starter data — don't change this
np.random.seed(7)
buildings = pd.DataFrame({
    'building': ['Tower A', 'Tower B', 'Annex', 'Depot'],
    'Jan': np.random.randint(150, 500, size=4),
    'Feb': np.random.randint(150, 500, size=4),
    'Mar': np.random.randint(150, 500, size=4),
    'Apr': np.random.randint(150, 500, size=4),
}).set_index('building')

# Your code here
energy_long = buildings.stack().to_frame('kwh')
energy_long['log_kwh'] = np.log(energy_long['kwh'])
energy_long['high_usage'] = np.where(energy_long['kwh']>300, True, False)
w = energy_long.unstack()
bm = energy_long.groupby('building')['kwh'].agg('mean')
bm.idxmax()

'Annex'

### Task 3: .apply() ✨ New

`.apply()` runs any function on a Series element-by-element, or on each row of a DataFrame.

```python
# On a Series — applied to each element:
df['col'].apply(lambda x: x * 2)
df['col'].apply(some_function)

# On a DataFrame — applied to each row (axis=1):
df.apply(lambda row: row['a'] + row['b'], axis=1)
df.apply(lambda row: 'High' if row['score'] > 80 else 'Low', axis=1)
```

Use `.apply()` when the logic is too complex for a single vectorized expression.

An e-commerce orders DataFrame has been created for you.

1. Add `total_price`: row-wise `unit_price * quantity` using `apply` with `axis=1`
2. Add `discounted_price`: row-wise `total_price * (1 - discount / 100)` — access `row['total_price']`
3. Add `price_tier`: apply a lambda directly to the `unit_price` Series — `'Premium'` if > 50, else `'Standard'`
4. Add `label`: row-wise — `'Big Spender'` if `total_price > 200` **and** `quantity > 3`, else `'Regular'`

In [48]:
# Starter data — don't change this
np.random.seed(15)
orders = pd.DataFrame({
    'order_id':   [f'ORD{i:03d}' for i in range(1, 9)],
    'product':    ['Keyboard', 'Monitor', 'Mouse', 'Webcam', 'Headset', 'Desk Lamp', 'USB Hub', 'Chair'],
    'unit_price': [35.0, 249.0, 22.0, 89.0, 65.0, 18.0, 29.0, 175.0],
    'quantity':   np.random.randint(1, 7, size=8),
    'discount':   np.random.choice([0, 5, 10, 15, 20], size=8),
})

# Your code here
orders['total_price']=orders.apply(lambda row: row['unit_price']*row['quantity'], axis =1)
orders['discounted_price'] = orders.apply(lambda row: row['total_price']*(1-row['discount']/100), axis=1)
orders['price_tier']= orders['unit_price'].apply(lambda x: 'Premium' if x>50 else 'Standard')
orders['label'] = orders.apply(lambda row: 'Big Spender' if (row['total_price']>200) & (row['quantity']>3) else 'Regular', axis=1)
orders

,order_id,product,unit_price,quantity,discount,total_price,discounted_price,price_tier,label
0,ORD001,Keyboard,35.0,1,5,35.0,33.25,Standard,Regular
1,ORD002,Monitor,249.0,6,0,1494.0,1494.00,Premium,Big Spender
2,ORD003,Mouse,22.0,5,10,110.0,99.00,Standard,Regular
3,ORD004,Webcam,89.0,6,20,534.0,427.20,Premium,Big Spender
4,ORD005,Headset,65.0,1,5,65.0,61.75,Premium,Regular
5,ORD006,Desk Lamp,18.0,5,15,90.0,76.50,Standard,Regular
6,ORD007,USB Hub,29.0,4,20,116.0,92.80,Standard,Regular
7,ORD008,Chair,175.0,4,20,700.0,560.00,Premium,Big Spender


### Task 4: .str + .rank()
A podcast episode catalog has been created for you.

1. Add `title_upper`: episode titles in uppercase
2. Filter to episodes where `title` contains `'Tech'` (case-insensitive)
3. Extract the numeric part of `episode_id` using `.str` slicing (e.g. `'EP007'` → `'007'`). Store as `ep_num`.
4. Add `global_rank`: rank by `plays`, highest first
5. Add `category_rank`: rank within each `category` by `plays`, highest first
6. Filter to the top-ranked episode per category (`category_rank == 1`)

In [20]:
# Starter data — don't change this
np.random.seed(29)
episodes = pd.DataFrame({
    'episode_id': [f'EP{i:03d}' for i in range(1, 9)],
    'title':      ['Tech Trends 2026', 'Deep Dives: AI', 'True Crime Hour',
                   'History Uncovered', 'Tech Talk Daily', 'Crime Files',
                   'Future of Tech', 'Ancient Mysteries'],
    'category':   ['Tech', 'Tech', 'Crime', 'History', 'Tech', 'Crime', 'Tech', 'History'],
    'plays':      np.random.randint(5_000, 80_000, size=8),
    'duration':   np.random.randint(20, 90, size=8),
})

# Your code here
episodes['title_upper'] = episodes['title'].str.upper()
t = episodes[episodes['title'].str.contains('Tech')]
episodes['ep_num'] = episodes['episode_id'].str[2:]
episodes['global_rank'] = episodes['plays'].rank(ascending=False)
episodes['category_rank'] = episodes.groupby('category')['plays'].rank(ascending=False)
episodes[episodes['category_rank']==1]

,episode_id,title,category,plays,duration,title_upper,ep_num,global_rank,category_rank
1,EP002,Deep Dives: AI,Tech,65962,84,DEEP DIVES: AI,002,1.0,1.0
5,EP006,Crime Files,Crime,57293,59,CRIME FILES,006,5.0,1.0
7,EP008,Ancient Mysteries,History,64342,87,ANCIENT MYSTERIES,008,2.0,1.0


---
## Level 3 — Aggregation

### Task 5: pd.merge() + groupby + transform
Two DataFrames have been created for you: `members` and `bookings`.

1. Inner join on `member_id`
2. Add `plan_avg_spend`: mean booking `amount` per `plan` using `transform`
3. Add `above_plan_avg`: `True` if `amount` > `plan_avg_spend`
4. Total booking `amount` per `city` — sort descending
5. Total booking `amount` per `plan` using groupby

In [24]:
# Starter data — don't change this
np.random.seed(61)
members = pd.DataFrame({
    'member_id': [f'M{i:03d}' for i in range(1, 9)],
    'city': ['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Sydney', 'Melbourne', 'Brisbane', 'Perth'],
    'plan': ['Gold', 'Silver', 'Bronze', 'Gold', 'Bronze', 'Gold', 'Silver', 'Silver'],
})

bookings = pd.DataFrame({
    'booking_id': [f'B{i:02d}' for i in range(1, 13)],
    'member_id':  np.random.choice([f'M{i:03d}' for i in range(1, 9)], size=12),
    'amount':     np.random.randint(30, 400, size=12),
    'month':      np.random.choice(['Jan', 'Feb', 'Mar'], size=12),
})

# Your code here
ij = pd.merge(
    members, 
    bookings, 
    how='inner',
    on='member_id'
)
ij['plan_avg_spend'] = ij.groupby('plan')['amount'].transform('mean')
ij['above_plan_avg'] = ij['amount']>ij['plan_avg_spend']
tm = ij.groupby('city')['amount'].sum().sort_values(ascending=False)
tm
ij.groupby('plan')['amount'].sum()

plan
Bronze     803
Gold       684
Silver    1246
Name: amount, dtype: int64

### Task 6: pd.concat() + pivot_table
Two monthly review DataFrames have been created for you.

1. Concatenate `reviews_jan` and `reviews_feb` vertically. Reset the index. Store as `reviews`.
2. Pivot: mean `rating` by `product` (rows) and `platform` (columns)
3. Pivot: count of `rating` by `product` (rows) and `month` (columns) — use `aggfunc='count'`
4. From the second pivot, find the product with the most reviews overall — sum across columns, then use `.idxmax()`

In [29]:
# Starter data — don't change this
np.random.seed(83)
reviews_jan = pd.DataFrame({
    'product':  np.random.choice(['Blender', 'Toaster', 'Kettle'], size=9),
    'platform': np.random.choice(['Web', 'App', 'In-store'], size=9),
    'rating':   np.random.randint(1, 6, size=9),
    'month':    'Jan',
})

reviews_feb = pd.DataFrame({
    'product':  np.random.choice(['Blender', 'Toaster', 'Kettle'], size=9),
    'platform': np.random.choice(['Web', 'App', 'In-store'], size=9),
    'rating':   np.random.randint(1, 6, size=9),
    'month':    'Feb',
})

# Your code here

reviews = pd.concat(
    [reviews_jan,reviews_feb],
    ignore_index=True
)
reviews.pivot_table(
    index='product',
    columns='platform',
    values='rating'
)

s = reviews.pivot_table(
    index='product',
    columns='month',
    values='rating',
    aggfunc='count'
)

s.sum(axis = 1).idxmax()


'Blender'

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Three DataFrames have been created for you: `patients`, `visits`, and `treatments`.

1. Convert `admit_date` to datetime. Add `days_since_admit` = days from `admit_date` to `2026-03-16`. Find mean and median with NumPy.
2. Inner join `visits` with `patients` on `patient_id`. Inner join the result with `treatments` on `treatment_id`. Drop nulls.
3. Use `.apply()` to add `risk_label` (row-wise): `'High'` if `age > 60` **and** `severity > 7`, `'Medium'` if either is true, else `'Low'`
4. Add `dept_avg_cost`: mean `cost` per `dept` using `transform`
5. Bin `cost` into 3 groups labelled `['Low', 'Medium', 'High']`. Count rows per group (use `observed=True`).
6. Pivot: total `cost` by `dept` (rows) and `month` (columns) — use `aggfunc='sum'`
7. Find the correlation between `age` and `cost` using NumPy

In [50]:
# Starter data — don't change this
np.random.seed(42)
n = 10

patients = pd.DataFrame({
    'patient_id': [f'P{i:03d}' for i in range(1, n + 1)],
    'age':        np.random.randint(18, 85, size=n),
    'dept':       np.random.choice(['Cardiology', 'Neurology', 'Oncology'], size=n),
    'admit_date': pd.date_range('2023-06-01', periods=n, freq='45D').strftime('%Y-%m-%d'),
})

treatments = pd.DataFrame({
    'treatment_id': [f'T{i:02d}' for i in range(1, 7)],
    'name':         ['Chemo', 'Radiation', 'Surgery', 'Therapy', 'Medication', 'Monitoring'],
    'cost':         [12000, 8500, 22000, 3500, 1800, 950],
})

visits = pd.DataFrame({
    'visit_id':     [f'V{i:02d}' for i in range(1, 25)],
    'patient_id':   np.random.choice([f'P{i:03d}' for i in range(1, n + 1)], size=24),
    'treatment_id': np.random.choice([f'T{i:02d}' for i in range(1, 7)], size=24),
    'severity':     np.random.randint(1, 11, size=24),
    'month':        np.random.choice(['Jan', 'Feb', 'Mar'], size=24),
})

# Your code here

patients['admit_date']=pd.to_datetime(patients['admit_date'])
patients['days_since_admit']= (pd.to_datetime('2026-03-16')-patients['admit_date']).dt.days
np.mean(patients['days_since_admit'])
np.median(patients['days_since_admit'])
ij = pd.merge(
    patients, 
    visits,
    how='inner',
    on='patient_id'
)
ij2 = pd.merge(
    ij,
    treatments, 
    on='treatment_id',
    how='inner'
).dropna()
ij2['risk_label'] = ij2.apply(
    lambda row: 'High' if (row['age']>60)&(row['severity']>7)
    else 'Medium' if (row['age']>60)|(row['severity']>7)
    else 'Low', 
    axis=1)
ij2['dept_avg_cost']= ij2.groupby('dept')['cost'].transform('mean')
ij2['g'] = pd.cut(ij2['cost'],3,labels=['Low','Medium','High'])
ij2.groupby('g',observed=True).agg('count')

p = ij2.pivot_table(
    index = 'dept',
    columns = 'month',
    values = 'cost',
    aggfunc = 'sum'
)
np.corrcoef(ij2['age'],ij2['cost'])
ij2

,patient_id,age,dept,admit_date,days_since_admit,visit_id,treatment_id,severity,month,name,cost,risk_label,dept_avg_cost,g
0,P002,32,Neurology,2023-07-16,974,V13,T06,9,Mar,Monitoring,950,Medium,3891.666667,Low
1,P002,32,Neurology,2023-07-16,974,V16,T02,8,Mar,Radiation,8500,Medium,3891.666667,Medium
2,P002,32,Neurology,2023-07-16,974,V21,T04,3,Feb,Therapy,3500,Low,3891.666667,Low
3,P003,78,Cardiology,2023-08-30,929,V02,T06,8,Mar,Monitoring,950,High,4436.666667,Low
4,P003,78,Cardiology,2023-08-30,929,V06,T04,8,Jan,Therapy,3500,High,4436.666667,Low
5,P003,78,Cardiology,2023-08-30,929,V08,T06,9,Jan,Monitoring,950,High,4436.666667,Low
6,P004,38,Cardiology,2023-10-14,884,V04,T04,2,Jan,Therapy,3500,Low,4436.666667,Low
7,P004,38,Cardiology,2023-10-14,884,V14,T05,8,Feb,Medication,1800,Medium,4436.666667,Low
8,P004,38,Cardiology,2023-10-14,884,V22,T06,3,Mar,Monitoring,950,Low,4436.666667,Low
9,P005,41,Neurology,2023-11-28,839,V07,T02,10,Feb,Radiation,8500,Medium,3891.666667,Medium
